# VirtualiZarr → Icechunk in Cloud Storage (Source Coop)

This notebook creates an Icechunk store on Source Coop containing virtual references to Copernicus Marine Service chlorophyll data. The Icechunk store references the original Copernicus data rather than copying it.

## Key Points
- Virtual references point to Copernicus cloudferro S3/HTTPS URLs
- No data duplication - only metadata stored in Icechunk
- Source Coop provides cloud storage for the Icechunk repository
- Users can access the data from anywhere with proper Copernicus credentials

## I need to patch icechunk

Until Source Coop allows CopyObject, I need to not backup the `repo` file when making icechunk commits. 

Make the edits to clone the icechunk repo so I have a local copy. The make edits to not use backup in icechunk/src/asset.rs. Then rebuild and pip install icechunk from latest wheel.
```
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh
source "$HOME/.cargo/env"
cd /home/jovyan/icechunk/icechunk-python
maturin build --release

find /home/jovyan/icechunk -path "*/target/wheels/*.whl"
```

In [2]:
!pip install -qU icechunk virtualizarr copernicusmarine xarray obstore obspec_utils

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xmip 0.7.2 requires cf_xarray>=0.6.0, which is not installed.
xmip 0.7.2 requires xarrayutils, which is not installed.
xmip 0.7.2 requires xgcm<0.7.0, which is not installed.


In [3]:
!python -m pip install -q --force-reinstall \
  /home/jovyan/icechunk/target/wheels/icechunk-2.1.1+patched-cp312-abi3-manylinux_2_39_x86_64.whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xmip 0.7.2 requires cf_xarray>=0.6.0, which is not installed.
xmip 0.7.2 requires xarrayutils, which is not installed.
xmip 0.7.2 requires xgcm<0.7.0, which is not installed.
numba 0.63.1 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.


In [4]:
import warnings
import time
from pathlib import Path
import json

import xarray as xr
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry

warnings.filterwarnings('ignore', category=UserWarning)
icechunk.__version__

'2.1.1+patched'

## Get Copernicus file URLs

Use `copernicusmarine` to get a list of files without downloading them.

In [2]:
# Get file list for 1997 (example - adjust dates as needed)
!copernicusmarine get \
  --dataset-id cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D \
  --dataset-version 202603 \
  --filter "*1997*.nc" \
  --create-file-list copernicus_files_sc.txt

INFO - 2026-07-25T03:48:07Z - Selected dataset version: "202603"
INFO - 2026-07-25T03:48:07Z - Selected dataset part: "default"
INFO - 2026-07-25T03:48:07Z - Listing files on remote server...
11it [00:05,  2.04it/s]
{
  "number_of_files_to_download": 0,
  "status": "002",
  "message": "The request created a file list and then stopped."
}


In [4]:
# Read and convert URLs
with open('copernicus_files_sc.txt', 'r') as f:
    s3_urls = [line.strip() for line in f if line.strip()]

s3_urls.sort()

COPERNICUS_ENDPOINT = "https://s3.waw3-1.cloudferro.com"
def s3_to_https(s3_url):
    if s3_url.startswith('s3://'):
        return f"{COPERNICUS_ENDPOINT}/{s3_url[5:]}"
    return s3_url

https_urls = [s3_to_https(url) for url in s3_urls]
print(f"Found {len(https_urls)} files")
print(f"First: {https_urls[0]}")
print(f"Last: {https_urls[-1]}")

Found 106 files
First: https://s3.waw3-1.cloudferro.com/mdl-native-16/native/OCEANCOLOUR_GLO_BGC_L3_MY_009_103/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D_202603/1997/09/19970904_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
Last: https://s3.waw3-1.cloudferro.com/mdl-native-16/native/OCEANCOLOUR_GLO_BGC_L3_MY_009_103/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D_202603/1997/12/19971231_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc


## Set up remote file configuration

Configure how to access the Copernicus cloudferro files.

In [5]:
# Create object-store handle for the REMOTE Copernicus files
url_prefix = f"{COPERNICUS_ENDPOINT}/"
store = from_url(url_prefix)
registry = ObjectStoreRegistry({url_prefix: store})
parser = HDFParser()

# Configure virtual chunk container
# This tells Icechunk where the actual data chunks live
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=url_prefix,
        store=icechunk.http_store(),
    )
)

print(f"✓ Remote storage configured for: {url_prefix}")

✓ Remote storage configured for: https://s3.waw3-1.cloudferro.com/


## Set up Source Coop storage

Source Coop provides cloud storage for the Icechunk repository.

**Important**: 
1. Get your Source Coop credentials from the 'View Credentials' link
2. Create a `source-creds.json` file (add to `.gitignore`)
3. Never hard-code credentials in notebooks

In [12]:
# Read Source Coop credentials from JSON file
with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

# Source Coop bucket information
# Adjust these to match your Source Coop organization and dataset
# https://source.coop/fish-pace/globcolour/<icechunk-name>
storage = icechunk.s3_storage(
    bucket="fish-pace",
    prefix=(
        "globcolour/"
        "test"
#        "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
    ),
    region=source_creds["region_name"],
    endpoint_url=source_creds["endpoint_url"],
    force_path_style=True,
    access_key_id=source_creds["aws_access_key_id"],
    secret_access_key=source_creds["aws_secret_access_key"],
    session_token=source_creds["aws_session_token"],
)

print("✓ Source Coop storage configured")

✓ Source Coop storage configured


In [13]:
# Create store if it is empty
try:
    repo = icechunk.Repository.create(storage, config)
    print("Created new Icechunk repo")
except Exception:
    repo = icechunk.Repository.open(storage, config=config)
    print("Opened existing Icechunk repo")

# Create a session
session = repo.writable_session(branch="main")

Created new Icechunk repo


## Write files to Icechunk

Process each file: virtualize and write/append to Icechunk.

In [14]:
# Process all files
commit_every = 5  # Commit every N files
total_added = 0
start = time.perf_counter()

for i, url in enumerate(https_urls):
    filename = Path(url).name
    
    print(f"[{i+1}/{len(https_urls)}] Processing {filename}...")
    
    # Open file virtually (no data download)
    vds = open_virtual_dataset(
        url=url,
        parser=parser,
        registry=registry,
        loadable_variables=['time', 'lat', 'lon', 'latitude', 'longitude'],
        decode_times=True,
    )
    
    # First file: create, subsequent: append
    if i == 0:
        vds.virtualize.to_icechunk(session.store)
    else:
        vds.virtualize.to_icechunk(session.store, append_dim="time")
    
    total_added += 1
    
    # Commit periodically
    if total_added % commit_every == 0:
        elapsed = time.perf_counter() - start
        snapshot_id = session.commit(f"Add through file {i + 1}")
        print(f"  Committed {total_added} files in {elapsed:.2f}s. Snapshot: {snapshot_id}")
        session = repo.writable_session("main")
        start = time.perf_counter()

# Final commit if needed
if total_added % commit_every != 0:
    snapshot_id = session.commit(f"Final commit: {total_added} files")
    print(f"Final commit: {snapshot_id}")

print(f"\n✓ Successfully added {total_added} files to Icechunk")

[1/106] Processing 19970904_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[2/106] Processing 19970906_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[3/106] Processing 19970909_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[4/106] Processing 19970910_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[5/106] Processing 19970915_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
  Committed 5 files in 17.10s. Snapshot: RR46WBYKJ288QNQSZPA0
[6/106] Processing 19970916_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[7/106] Processing 19970918_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[8/106] Processing 19970919_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[9/106] Processing 19970920_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
[10/106] Processing 19970921_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc...
  Committed 10 files in 22.95s. Snapshot: 5XA9R67GNPH97FWWE9RG
[11/106] Processing 19970922_cmems

## Read in the icechunk

✓ Created Icechunk repository on Source Coop
✓ Stored virtual references to Copernicus cloudferro data
✓ No data duplication - only metadata in Icechunk


In [16]:
import icechunk as ic
import xarray as xr

url = (
    "https://data.source.coop/fish-pace/globcolour/"
    "cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
)
url = (
    "https://data.source.coop/fish-pace/globcolour/"
    "test"
)
storage = ic.http_storage(url)

repo = ic.Repository.open(storage)
containers = repo.config.virtual_chunk_containers or []
store = ic.Repository.open(
    storage,
    authorize_virtual_chunk_access={
        prefix: icechunk.credentials.HttpAccess
        for prefix in containers
    },
).readonly_session("main").store

ds = xr.open_zarr(
    store,
    consolidated=False,
    chunks={},
)

ds


<xarray.Dataset> Size: 479GB
Dimensions:              (time: 106, lat: 4320, lon: 8640)
Coordinates:
  * time                 (time) datetime64[ns] 848B 1997-09-04 ... 1997-12-31
  * lat                  (lat) float32 17kB 89.98 89.94 89.9 ... -89.94 -89.98
  * lon                  (lon) float32 35kB -180.0 -179.9 -179.9 ... 179.9 180.0
Data variables: (12/21)
    DINO                 (time, lat, lon) float32 16GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    GREEN                (time, lat, lon) float32 16GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    CHL                  (time, lat, lon) float32 16GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    DINO_uncertainty     (time, lat, lon) float64 32GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    GREEN_uncertainty    (time, lat, lon) float64 32GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    DIATO                (time, lat, lon) float32 16GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    ...                   ...
    NANO_uncertainty     (time, lat, lon) float64 32GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    PICO                 (time, lat, lon) float32 16GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    MICRO                (time, lat, lon) float32 16GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    flags                (time, lat, lon) int8 4GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    PROCHLO              (time, lat, lon) float32 16GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    MICRO_uncertainty    (time, lat, lon) float64 32GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
Attributes: (12/91)
    lon_step:                        0.0416666679084301
    keywords:                        EARTH SCIENCE > OCEANS > OCEAN CHEMISTRY...
    cmems_product_id:                OCEANCOLOUR_GLO_BGC_L3_MY_009_103
    lat_step:                        0.0416666679084301
    grid_mapping:                    Equirectangular
    title:                           cmems_obs-oc_glo_bgc-plankton_my_l3-mult...
    ...                              ...
    contact:                         servicedesk.cmems@acri-st.fr
    westernmost_longitude:           -180.0
    geospatial_vertical_positive:    up
    date_created:                    2023-10-13T20:37:28Z
    nb_valid_bins:                   2043353
    pct_valid_bins:                  5.474518282750343

## Full set of years

In [17]:
# Get file list for 1997 (example - adjust dates as needed)
!copernicusmarine get \
  --dataset-id cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D \
  --dataset-version 202603 \
  --filter "*.nc" \
  --create-file-list copernicus_files_all.txt

INFO - 2026-07-25T04:31:52Z - Selected dataset version: "202603"
INFO - 2026-07-25T04:31:52Z - Selected dataset part: "default"
INFO - 2026-07-25T04:31:52Z - Listing files on remote server...
11it [00:05,  1.98it/s]
{
  "number_of_files_to_download": 0,
  "status": "002",
  "message": "The request created a file list and then stopped."
}


In [3]:
# Read and convert URLs
with open('copernicus_files_all.txt', 'r') as f:
    s3_urls = [line.strip() for line in f if line.strip()]

s3_urls.sort()

COPERNICUS_ENDPOINT = "https://s3.waw3-1.cloudferro.com"
def s3_to_https(s3_url):
    if s3_url.startswith('s3://'):
        return f"{COPERNICUS_ENDPOINT}/{s3_url[5:]}"
    return s3_url

https_urls = [s3_to_https(url) for url in s3_urls]
print(f"Found {len(https_urls)} files")
print(f"First: {https_urls[0]}")
print(f"Last: {https_urls[-1]}")

Found 10490 files
First: https://s3.waw3-1.cloudferro.com/mdl-native-16/native/OCEANCOLOUR_GLO_BGC_L3_MY_009_103/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D_202603/1997/09/19970904_cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D.nc
Last: https://s3.waw3-1.cloudferro.com/mdl-native-16/native/OCEANCOLOUR_GLO_BGC_L3_MY_009_103/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D_202603/2026/07/20260717_cmems_obs-oc_glo_bgc-plankton_myint_l3-multi-4km_P1D.nc


In [4]:
# Create object-store handle for the REMOTE Copernicus files
url_prefix = f"{COPERNICUS_ENDPOINT}/"
store = from_url(url_prefix)
registry = ObjectStoreRegistry({url_prefix: store})
parser = HDFParser()

# Configure virtual chunk container
# This tells Icechunk where the actual data chunks live
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=url_prefix,
        store=icechunk.http_store(),
    )
)

print(f"✓ Remote storage configured for: {url_prefix}")

✓ Remote storage configured for: https://s3.waw3-1.cloudferro.com/


In [5]:
import json
from pathlib import Path
import time
from datetime import datetime, timezone, timedelta

import icechunk


def _get_source_creds_expiration(
    creds_file,
    assumed_ttl_minutes=60,
):
    """
    Get expiration from the file modification time since json has no expiration
    """

    mtime = datetime.fromtimestamp(
        Path(creds_file).stat().st_mtime,
        tz=timezone.utc,
    )

    expiration = mtime + timedelta(minutes=assumed_ttl_minutes)
    return expiration

def open_source_icechunk_repo(
    creds_file="globcolour-source-creds.json",
    bucket="fish-pace",
    prefix="globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D",
    config=None,
    min_minutes_left=15,
    assumed_ttl_minutes=60,
    create_if_missing=True,
    verbose=True,
    check_expiration=True,
):
    """
    Open or create an Icechunk repo using Source Cooperative temporary credentials.

    If the creds file has no explicit expiration, estimate expiration from:
        file modification time + assumed_ttl_minutes

    If check_expiration=True, raises RuntimeError if less than min_minutes_left remain.
    """

    creds_file = Path(creds_file)

    with creds_file.open() as f:
        source_creds = json.load(f)

    expiration = _get_source_creds_expiration(
        creds_file=creds_file,
        assumed_ttl_minutes=assumed_ttl_minutes,
    )

    now = datetime.now(timezone.utc)
    time_left = expiration - now

    if check_expiration and time_left < timedelta(minutes=0):
        print(
            f"Stopping cleanly. Source credentials expired. "
            f"Refresh {creds_file} and set start_index."
        )
        return None, None, None, time_left

    storage = icechunk.s3_storage(
        bucket=bucket,
        prefix=prefix,
        region=source_creds["region_name"],
        endpoint_url=source_creds["endpoint_url"],
        force_path_style=True,
        access_key_id=source_creds["aws_access_key_id"],
        secret_access_key=source_creds["aws_secret_access_key"],
        session_token=source_creds["aws_session_token"],
    )

    if create_if_missing:
        try:
            repo = icechunk.Repository.create(storage, config)
            if verbose:
                print("Created new Icechunk repo")
        except Exception:
            repo = icechunk.Repository.open(storage, config=config)
            if verbose:
                print("Opened existing Icechunk repo")
    else:
        repo = icechunk.Repository.open(storage, config=config)
        if verbose:
            print("Opened existing Icechunk repo")

    if verbose:
        print(f"Time remaining on token: {time_left}")

    return repo, storage, source_creds, time_left

In [6]:
import time
from datetime import timedelta
from pathlib import Path

from virtualizarr import open_virtual_dataset


def wait_for_fresh_repo(
    *,
    creds_file="globcolour-source-creds.json",
    min_minutes_left=15,
    config=None,
    verbose=True,
):
    while True:
        repo, storage, source_creds, time_left = open_source_icechunk_repo(
            creds_file=creds_file,
            create_if_missing=True,
            config=config,
            check_expiration=True,
            verbose=False,
        )

        if time_left >= timedelta(minutes=min_minutes_left):
            if verbose:
                print(f"Token okay. Time remaining: {time_left}")
            return repo, storage, source_creds, time_left

        print(
            f"Source credentials expire in about {time_left}. "
            f"Refresh {creds_file} before continuing."
        )

        try:
            answer = input(
                "Enter y after refreshing the token, or n to stop: "
            ).strip().lower()
        except KeyboardInterrupt:
            print("Input interrupted. Stopping cleanly.")
            return None, None, None, time_left

        if answer == "y":
            continue

        if answer == "n":
            print(
                "Stopping. Refresh the token and rerun starting "
                "at the last committed file."
            )
            return None, None, None, time_left

        print("Please enter y or n.")


def write_globcolour_to_icechunk(
    https_urls,
    *,
    commit_every=5,
    start_index=0,
    branch="main",
    group=None,
    config=None,  # required
    creds_file="globcolour-source-creds.json",
    min_minutes_left=15,
    parser=parser,
    registry=registry,
):
    """
    Virtualize GlobColour NetCDF files and append them along time.

    Parameters
    ----------
    https_urls
        Ordered list of GlobColour NetCDF URLs.
    commit_every
        Number of files per Icechunk commit.
    start_index
        Index of the first unprocessed URL. Use this to resume after stopping.
    branch
        Icechunk branch to write to.
    group
        Optional Zarr group within the Icechunk repository.
    config
        Icechunk RepositoryConfig, including the HTTP virtual chunk container.
    creds_file
        Source Cooperative temporary credentials JSON.
    min_minutes_left
        Require at least this much token lifetime before opening a new session.
    parser, registry
        VirtualiZarr parser and ObjectStoreRegistry.
    """
    if config is None:
        raise ValueError("config is required")

    total_added = start_index

    repo, storage, source_creds, time_left = wait_for_fresh_repo(
        creds_file=creds_file,
        min_minutes_left=min_minutes_left,
        config=config,
        verbose=False,
    )

    if repo is None:
        return total_added

    print(f"Token time remaining: {time_left}")

    session = repo.writable_session(branch)
    batch_start = time.perf_counter()
    files_in_batch = 0

    for i, url in enumerate(
        https_urls[start_index:],
        start=start_index,
    ):
        filename = Path(url).name
        # print(f"[{i + 1}/{len(https_urls)}] Processing {filename}...")

        # GlobColour files already contain their time coordinate.
        vds = open_virtual_dataset(
            url=url,
            parser=parser,
            registry=registry,
            loadable_variables=[
                "time",
                "lat",
                "lon",
                "latitude",
                "longitude",
            ],
            decode_times=True,
        )

        write_kwargs = {}

        if group is not None:
            write_kwargs["group"] = group

        # Create the hierarchy for the first file in the complete dataset.
        # Every later file appends along the existing time dimension.
        if total_added == 0:
            vds.virtualize.to_icechunk(
                session.store,
                **write_kwargs,
            )
        else:
            vds.virtualize.to_icechunk(
                session.store,
                append_dim="time",
                **write_kwargs,
            )

        total_added += 1
        files_in_batch += 1

        if files_in_batch == commit_every:
            elapsed = time.perf_counter() - batch_start

            snapshot_id = session.commit(
                f"Add through file {i + 1}"
            )

            print(
                f"Committed through file {i + 1} "
                f"({total_added} total files) in {elapsed:.2f} seconds. "
                f"Snapshot: {snapshot_id}"
            )

            # Reopen the repository with a fresh token if necessary.
            repo, storage, source_creds, time_left = wait_for_fresh_repo(
                creds_file=creds_file,
                min_minutes_left=min_minutes_left,
                config=config,
                verbose=False,
            )

            if repo is None:
                print(
                    f"Stopped after file {i + 1}. "
                    f"Resume with start_index={total_added}."
                )
                return total_added

            print(f"Token time remaining: {time_left}")

            session = repo.writable_session(branch)
            batch_start = time.perf_counter()
            files_in_batch = 0

    # Commit the final partial batch.
    if files_in_batch > 0:
        elapsed = time.perf_counter() - batch_start

        snapshot_id = session.commit(
            f"Add through file {total_added}"
        )

        print(
            f"Final commit through file {total_added} "
            f"in {elapsed:.2f} seconds. "
            f"Snapshot: {snapshot_id}"
        )

    print(
        f"\n✓ Successfully added {total_added - start_index} new files "
        f"({total_added} total files processed)"
    )

    return total_added

In [ ]:
total_added = write_globcolour_to_icechunk(
    https_urls,
    commit_every=10,
    start_index=0,
    config=config,
    creds_file="globcolour-source-creds.json",
    min_minutes_left=10
)

In [ ]:
total_added = write_globcolour_to_icechunk(
    https_urls,
    commit_every=10,
    start_index=1810,
    config=config,
    creds_file="globcolour-source-creds.json",
    min_minutes_left=5
)

Token time remaining: 0:58:37.109629


## Troubleshooting

In [10]:
# List everything
import boto3
import json

with open("globcolour-source-creds.json") as f:
    creds = json.load(f)

s3 = boto3.client(
    "s3",
    endpoint_url=creds["endpoint_url"],
    region_name=creds["region_name"],
    aws_access_key_id=creds["aws_access_key_id"],
    aws_secret_access_key=creds["aws_secret_access_key"],
    aws_session_token=creds["aws_session_token"],
)

bucket = "fish-pace"
prefix = "globcolour/"
DRY_RUN = True

paginator = s3.get_paginator("list_objects_v2")

keys = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        keys.append(obj["Key"])
        print(f'{obj["Size"]:>12,}  {obj["Key"]}')

print(f"\nFound {len(keys):,} objects under s3://{bucket}/{prefix}")


Found 0 objects under s3://fish-pace/globcolour/


In [8]:
%%script false --no-raise
# Delete everything
import json
import boto3

with open("globcolour-source-creds.json") as f:
    source_creds = json.load(f)

bucket = "fish-pace"
prefix = "globcolour"

s3 = boto3.client(
    "s3",
    endpoint_url=source_creds["endpoint_url"],
    region_name=source_creds["region_name"],
    aws_access_key_id=source_creds["aws_access_key_id"],
    aws_secret_access_key=source_creds["aws_secret_access_key"],
    aws_session_token=source_creds["aws_session_token"],
)

def delete_s3_prefix(bucket, prefix):
    prefix = prefix.strip("/") + "/"

    paginator = s3.get_paginator("list_objects_v2")

    # Gather keys first so changing the listing while paginating
    # does not cause objects to be skipped.
    keys = []

    for page in paginator.paginate(
        Bucket=bucket,
        Prefix=prefix,
    ):
        keys.extend(
            obj["Key"]
            for obj in page.get("Contents", [])
        )

    print(f"Found {len(keys)} objects under s3://{bucket}/{prefix}")

    for i, key in enumerate(keys, start=1):
        s3.delete_object(
            Bucket=bucket,
            Key=key,
        )

        if i % 100 == 0 or i == len(keys):
            print(f"Deleted {i} of {len(keys)} objects")

    print(f"Deleted everything under s3://{bucket}/{prefix}")


delete_s3_prefix(bucket, prefix)

Found 329 objects under s3://fish-pace/globcolour/
Deleted 100 of 329 objects
Deleted 200 of 329 objects
Deleted 300 of 329 objects
Deleted 329 of 329 objects
Deleted everything under s3://fish-pace/globcolour/


In [9]:
# Check it is indeed empty
response = s3.list_objects_v2(
    Bucket="fish-pace",
    Prefix="globcolour/",
    MaxKeys=10,
)

print(response.get("Contents", []))

[]
